# Setup

Imports and such:

In [46]:
from importlib import reload

import numpy as np
import pyvista as pv

import skyvista as sv

import carlee_tools as ct

 Read in our example dataset

In [2]:
# reload(sv.examples)
ds = sv.load_example_storm_data()
ds

<xarray.Dataset> Size: 87MB
Dimensions:       (time: 1, z: 108, y: 201, x: 201)
Coordinates:
  * time          (time) datetime64[ns] 8B 1991-04-26T22:40:00
    t_minutes     (time) float64 8B ...
  * z             (z) float64 864B -11.63 12.2 38.42 ... 2.258e+04 2.283e+04
  * y             (y) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
  * x             (x) float64 2kB 6e+04 6.05e+04 6.1e+04 ... 1.595e+05 1.6e+05
Data variables:
    UC            (time, z, y, x) float32 17MB ...
    VC            (time, z, y, x) float32 17MB ...
    WC            (time, z, y, x) float32 17MB ...
    THETA         (time, z, y, x) float32 17MB ...
    R_condensate  (time, z, y, x) float32 17MB ...
    ACCPR         (time, y, x) float32 162kB ...

UC, VC, and WC are the x/y/z components of the wind, THETA is the potential temperature, R_condensate is the mass mixing ratio of all hydrometeors, and ACCPR is the cumulative amount of rain in a surface gridpoint.

# Ways to use skyvista from simplest to most complicated

## Simplest: `quick_plot`

### tl;dr

In [67]:
reload(sv.plotter)
reload(sv)

# Read in example data
ds = sv.load_example_storm_data()

# Make a new plotter/scene
plotter = sv.initialize_plotter()
# Exagerate the z scale
plotter.set_scale(zscale=3)

# Calculate variable we'll use as the cold pool
ds["theta_deficit"] = (ds["THETA"].mean(["x", "y"]) - ds["THETA"]).where(
    ds["z"] <= 3000
)

# Make separate updraft and downdraft variables
ds["updraft"] = ds["WC"].where(ds["WC"] > 0)
ds["downdraft"] = -ds["WC"].where(ds["WC"] < 0)

# Make base-state relative winds
for wind_var in ['UC', 'VC', 'WC']:
    ds[f"{wind_var}_bsr"] = ds[wind_var] - ds[wind_var].mean(['x', 'y'])

# Now actually create the scene
_ = sv.quick_plot(
    # Pass the plotter we created
    plotter=plotter,
    simulation_ds=ds,
    contours={
        # Updraft contours
        "updraft": {
            "opacity": 0.4,
            "isosurfaces": np.arange(7, 25, 2),
            "cmap": ct.plotting.shifted_colormap("Greens", (0.3, 1.0)),
        },
        # Downdraft contours
        "downdraft": {
            "opacity": 0.4,
            "isosurfaces": np.arange(2, 8, 2),
            "cmap": ct.plotting.shifted_colormap("Oranges", (0.3, 1.0)),
        },
        # Cold pool
        "theta_deficit": {"opacity": 0.5, "isosurfaces": [1, 2], "cmap": "Blues"},
    },
    # 2D surface of accumulated rain
    slices_2d={"ACCPR": {"cmap": "Blues"}},
    # Wind vectors
    vectors={
        "wind": {"u": "UC_bsr", "v": "VC_bsr", "w": "WC_bsr", "factor": 500, "tolerance": 0.05}
    },
    # Volume for cloud
    volumes={
        "R_condensate": {
            "cmap": "Blues",
            "opacity": sv.presets.cloud_opacity_function,
            "clim": [1e-5, ds["R_condensate"].max().values],
        }
    },
)


# Set the camera position
plotter.camera_position = pv.CameraPosition(
    position=(201249.4941441632, 201249.4941441632, 49336.84095394194),
    focal_point=(110000.0, 110000.0, 18920.34290599823),
    viewup=(0.0, 0.0, 1.0),
)

Widget(value='<iframe src="http://localhost:35933/index.html?ui=P_0x7f2d75d2df90_43&reconnect=auto" class="pyv…

`quick_plot` is the simplest way to take a quick look at something, or setup a simple scene
from a single dataset. Suppose we want to see the general shape and location of our storm's updraft:

In [3]:
# Call quick_plot
# The `_ =` just makes it so jupyter doesn't print the returned meshes, since we don't
# need them here
_ = sv.quick_plot(
    # Pass your (potentially non-uniformly) gridded data
    simulation_ds=ds,
    contours={"WC": {}},
)

2026-03-18 19:26:42.471 (   2.851s) [    7F30BE648740]vtkXOpenGLRenderWindow.:1460  WARN| bad X server connection. DISPLAY=:99.0


Widget(value='<iframe src="http://localhost:35933/index.html?ui=P_0x7f2e6100d550_0&reconnect=auto" class="pyvi…

Exception raised
ClientConnectionResetError('Cannot write to closing transport')
Traceback (most recent call last):
  File "/home/cmdavis4/projects/carlee/skyvista/.pixi/envs/default/lib/python3.14/site-packages/wslink/protocol.py", line 322, in onCompleteMessage
    await self.sendWrappedMessage(
        rpcid, results, method=methodName, client_id=client_id
    )
  File "/home/cmdavis4/projects/carlee/skyvista/.pixi/envs/default/lib/python3.14/site-packages/wslink/protocol.py", line 426, in sendWrappedMessage
    await ws.send_bytes(chunk)
  File "/home/cmdavis4/projects/carlee/skyvista/.pixi/envs/default/lib/python3.14/site-packages/aiohttp/web_ws.py", line 426, in send_bytes
    await self._writer.send_frame(data, WSMsgType.BINARY, compress=compress)
  File "/home/cmdavis4/projects/carlee/skyvista/.pixi/envs/default/lib/python3.14/site-packages/aiohttp/_websocket/writer.py", line 110, in send_frame
    await asyncio.shield(send_task)
  File "/home/cmdavis4/projects/carlee/skyvista/

Note the form of what we pass to `quick_plot`: 
- Keyword argument of the type of object we want to create in the scene (contours of isosurfaces)
- Value of this argument is a dict where:
  - Each key is the name of a variable in the dataset and,
  - Each value is a dict of plotting settings for that variable
  
In the example above, we passed an empty dict for the `WC` plotting settings to just use all default options.

This worked, but the visualization isn't that useful, since we can only see the outermost isosurface. Nonetheless, we can see that the scene is interactive and shows the general shape of the storm and its cold pool. We can give all of the isosurfaces a uniform semi-transparent opacity to make this more useful:

In [4]:
_ = sv.quick_plot(
    simulation_ds=ds,
    contours={"WC": {"opacity": 0.4}},
)

Widget(value='<iframe src="http://localhost:35933/index.html?ui=P_0x7f2d76821310_1&reconnect=auto" class="pyvi…

Now we can see the core of the upward vertical motion much more clearly. However, we often want to see only the updraft, not the downdraft. A few ways of achieving this:

In [ ]:
_ = sv.quick_plot(
    # Simplest option: manipulate and/or subset your data
    simulation_ds=ds.where(ds["WC"] > 0),
    contours={"WC": {"opacity": 0.4}},
)

Widget(value='<iframe src="http://localhost:35933/index.html?ui=P_0x7f2da06f0050_2&reconnect=auto" class="pyvi…

In [8]:
# For contours, another option is specifying isosurface values directly via the
# "isosurfaces" keyword argument
# `where` calls can be slow on large xarray Datasets, so this is probably preferable
# if the subsetting you want can be easily expressed this way

_ = sv.quick_plot(
    simulation_ds=ds,
    contours={"WC": {"opacity": 0.4, "isosurfaces": np.arange(1, 25, 2)}},
)

Widget(value='<iframe src="http://localhost:35933/index.html?ui=P_0x7f2da06f2fd0_4&reconnect=auto" class="pyvi…

We'll go with the second approach for now. For a continuous variable like velocity, we may want to color this such that only the intensity of the color corresponds with the value, rather than changing the hue as in a colormap like `viridis`:

In [9]:
# For contours, another option is specifying isosurface values directly via the
# "isosurfaces" keyword argument
# `where` calls can be slow on large xarray Datasets, so this is probably preferable
# if the subsetting you want can be easily expressed this way

_ = sv.quick_plot(
    simulation_ds=ds,
    contours={
        "WC": {"opacity": 0.4, "isosurfaces": np.arange(1, 25, 2), "cmap": "Greens"}
    },
)

Widget(value='<iframe src="http://localhost:35933/index.html?ui=P_0x7f2da06f3890_5&reconnect=auto" class="pyvi…

Looking much better. It's simple to show multiple variables; let's add a 2D surface of the accumulated rainfall with the `slices_2d` argument:

In [ ]:
_ = sv.quick_plot(
    simulation_ds=ds,
    contours={
        "WC": {"opacity": 0.4, "isosurfaces": np.arange(1, 25, 2), "cmap": "Greens"}
    },
    slices_2d={"ACCPR": {"cmap": "Blues"}},
)

Widget(value='<iframe src="http://localhost:35933/index.html?ui=P_0x7f2e4d98ae90_7&reconnect=auto" class="pyvi…

And the cold pool, which we'll calculate as the low-level potential temperature deficit relative to the layer-average value:

In [ ]:
ds["theta_lr"] = (ds["THETA"].mean(["x", "y"]) - ds["THETA"]).where(ds["z"] <= 3000)

_ = sv.quick_plot(
    simulation_ds=ds,
    contours={
        "WC": {"opacity": 0.4, "isosurfaces": np.arange(1, 25, 2), "cmap": "Greens"},
        "theta_lr": {"opacity": 0.5, "isosurfaces": [1, 2], "cmap": "Blues"},
    },
    slices_2d={"ACCPR": {"cmap": "Blues"}},
)

Widget(value='<iframe src="http://localhost:35933/index.html?ui=P_0x7f2d98394410_15&reconnect=auto" class="pyv…

Nice, but sort of hard to see. Let's do two things: first, let's limit the updraft contours to larger values of W to see it more clearly and remove some of the noise from the vertical motion at the leading edge of the cold pool. Second, let's zoom in. The way we can hardcode this is with a `pv.Plotter` object. Under the hood of skyvista, all of the objects being created are passed to a `Plotter` object from pyvista, which is like their concept of a "scene". The camera position can be gotten or set via the `.camera_position` argument. Skyvista has a convenience function for creating a plotter with sensible defaults (which `quick_plot` was already using under the hood), and this can be used to get a hold of the plotter explicitly like so:

In [34]:
# Create the plotter object
plotter = sv.initialize_plotter()

_ = sv.quick_plot(
    # Pass our plotter into quick_plot
    plotter=plotter,
    simulation_ds=ds,
    contours={
        "WC": {"opacity": 0.4, "isosurfaces": np.arange(5, 25, 2), "cmap": "Greens"},
        "theta_lr": {"opacity": 0.5, "isosurfaces": [1, 2], "cmap": "Blues"},
    },
    slices_2d={"ACCPR": {"cmap": "Blues"}},
)

Widget(value='<iframe src="http://localhost:35933/index.html?ui=P_0x7f2e4da2ee90_22&reconnect=auto" class="pyv…

Now we can get the camera position from the plotter object. Try running all of the above cells up to this point, then alternate between moving around in the interactive scene directly above this cell and running the following cell, and see how the value it returns changes:

In [40]:
plotter.camera_position

CameraPosition(position=(201249.4941441632, 201249.4941441632, 49336.84095394194),
               focal_point=(110000.0, 110000.0, 18920.34290599823),
               viewup=(0.0, 0.0, 1.0))

Now if we set the `.camera_position` attribute of a plotter to this value, anything we plot will start at this camera angle:

In [ ]:
# Create another new plotter object
plotter = sv.initialize_plotter()

# Create the scene
_ = sv.quick_plot(
    # Pass our plotter into quick_plot
    plotter=plotter,
    simulation_ds=ds,
    contours={
        "WC": {"opacity": 0.4, "isosurfaces": np.arange(5, 25, 2), "cmap": "Greens"},
        "theta_lr": {"opacity": 0.5, "isosurfaces": [1, 2], "cmap": "Blues"},
    },
    slices_2d={"ACCPR": {"cmap": "Blues"}},
)

# Set the camera position, based on the value we got by interactively moving around
# the figure above and copy-pasting the output of the code cell above this one
# You can set the camera position before creating the scene, but this sometimes
# causes nothing to show up until you pan or zoom the scene very slightly, which
# seems to force a render; so this way is just cleaner
plotter.camera_position = pv.CameraPosition(
    position=(201249.4941441632, 201249.4941441632, 49336.84095394194),
    focal_point=(110000.0, 110000.0, 18920.34290599823),
    viewup=(0.0, 0.0, 1.0),
)

Widget(value='<iframe src="http://localhost:35933/index.html?ui=P_0x7f2d8ff51a90_28&reconnect=auto" class="pyv…

Handling the `Plotter` object directly opens up further options. Let's say we want to show both the updraft and downdraft, but with different colormaps and max/min values. Diverging colormaps with different max/min magnitudees are tricky in pyvista; there is no analog to the `TwoSlopeNorm` that can be used to accomplish this in matplotlib. You could create separate variables for each, but we already saw that `.where` operations can be expensive on large xarray datasets. We could create two sets of contours with different isosurface values and colormaps using `quick_plot` like we have been, but we can't pass a dictionary with two "WC" keys. But with the plotter, we can just call `quick_plot` twice:

In [55]:
# Make a new plotter/scene
plotter = sv.initialize_plotter()

# Call quick_plot like we have been, with the addition of the `show=False` to make
# it delay rendering until we've added the downdraft (show=True by default):
_ = sv.quick_plot(
    # Reusing our existing plotter
    plotter=plotter,
    simulation_ds=ds,
    contours={
        "WC": {"opacity": 0.4, "isosurfaces": np.arange(5, 25, 2), "cmap": "Greens"},
        "theta_lr": {"opacity": 0.5, "isosurfaces": [1, 2], "cmap": "Blues"},
    },
    slices_2d={"ACCPR": {"cmap": "Blues"}},
    show=False,
)

# Now call it again for the downdraft:
_ = sv.quick_plot(
    # Reusing our existing plotter
    plotter=plotter,
    simulation_ds=ds,
    contours={
        "WC": {
            "opacity": 0.4,
            "isosurfaces": np.arange(-8, -2, 2),
            "cmap": "Reds_r",
        },
    },
)

plotter.camera_position = pv.CameraPosition(
    position=(201249.4941441632, 201249.4941441632, 49336.84095394194),
    focal_point=(110000.0, 110000.0, 18920.34290599823),
    viewup=(0.0, 0.0, 1.0),
)

Widget(value='<iframe src="http://localhost:35933/index.html?ui=P_0x7f2d7e1ed950_37&reconnect=auto" class="pyv…

A common problem when plotting contours in matplotlib that is especially prevalent in pyvista is that the sequential colormaps included with matplotlib all start at very light colors, so the lowest contour is sometimes very nearly white, as we can see in the above. A tool for dealing with this is the `shifted_colormap` function from the `carlee_tools` package:

In [54]:
plotter = sv.initialize_plotter()

# Note the colormap we use for WC here
_ = sv.quick_plot(
    plotter=plotter,
    simulation_ds=ds,
    contours={
        "WC": {
            "opacity": 0.4,
            "isosurfaces": np.arange(5, 25, 2),
            # Use carlee_tools to create a colormap that starts 30% of the way
            # through the normal matplotlib `Greens` colormap
            "cmap": ct.plotting.shifted_colormap("Greens", (0.3, 1)),
        },
        "theta_lr": {"opacity": 0.5, "isosurfaces": [1, 2], "cmap": "Blues"},
    },
    slices_2d={"ACCPR": {"cmap": "Blues"}},
    # show=False,
)

# _ = sv.quick_plot(
#     plotter=plotter,
#     simulation_ds=ds,
#     contours={
#         "WC": {
#             "opacity": 0.4,
#             "isosurfaces": np.arange(-8, -2, 2),
#             # Analogously to the updraft, but for a reversed colormap
#             "cmap": ct.plotting.shifted_colormap("Reds_r", (0, 0.5)),
#         },
#     },
# )

plotter.camera_position = pv.CameraPosition(
    position=(201249.4941441632, 201249.4941441632, 49336.84095394194),
    focal_point=(110000.0, 110000.0, 18920.34290599823),
    viewup=(0.0, 0.0, 1.0),
)

Widget(value='<iframe src="http://localhost:35933/index.html?ui=P_0x7f2d7e612e90_36&reconnect=auto" class="pyv…

In [ ]:
ds.sel({
    "x": slice(None, None, xy_stride),
    "y": slice(None, None, xy_stride),
    "z": slice(None, None, z_stride),
})

NameError: name 'xy_stride' is not defined

In [ ]:
meshes[0].values[1]

PVMesh(varspec=PVVectorSpec(varname='wind', name='type-vector_category-wind_varname-wind', scalar_bar=False, create_mesh_kwargs={'factor': 0.01}, add_mesh_kwargs={}, empty_ok=False, u_varname='UC', v_varname='VC', w_varname='WC'), name='dt-19910426224000_type-vector_category-wind_varname-wind', time=np.datetime64('1991-04-26T22:40:00.000000000'), mesh=PolyData (0x7fda3c716140)
  N Cells:    13365
  N Points:   27621
  N Strips:   0
  X Bounds:   6.000e+04, 1.600e+05
  Y Bounds:   6.000e+04, 1.600e+05
  Z Bounds:   -1.166e+01, 2.108e+04
  N Arrays:   5, actor={(0, 0): Actor (0x7fda3c716aa0)
  Center:                     (110000.0390625, 110000.0, 31608.034744262695)
  Pickable:                   True
  Position:                   (0.0, 0.0, 0.0)
  Scale:                      (1.0, 1.0, 3.0)
  Visible:                    True
  X Bounds                    6.000E+04, 1.600E+05
  Y Bounds                    6.000E+04, 1.600E+05
  Z Bounds                    -3.499E+01, 6.325E+04
  User mat

In [ ]:
ds.isel({
    "x": slice(None, None, xy_stride),
    "y": slice(None, None, xy_stride),
    "z": slice(None, None, z_stride),
})["WC"].max()

<xarray.DataArray 'WC' ()> Size: 4B
array(15.250919, dtype=float32)
Attributes:
    dimensions:   nx,ny,nz
    units:        m/s
    description:  Current W wind component

In [ ]:
meshes[0].values[0]

PVMesh(varspec=PVVolumeSpec(varname='WC', name='type-unknown_category-WC_varname-WC', scalar_bar=False, create_mesh_kwargs={}, add_mesh_kwargs={'clim': [1, 10]}, empty_ok=False, individual_meshes=False), name='dt-19910426224000_type-unknown_category-WC_varname-WC', time=np.datetime64('1991-04-26T22:40:00.000000000'), mesh=RectilinearGrid (0x7fda410406a0)
  N Cells:      4280000
  N Points:     4363308
  X Bounds:     6.000e+04, 1.600e+05
  Y Bounds:     6.000e+04, 1.600e+05
  Z Bounds:     -1.163e+01, 2.283e+04
  Dimensions:   201, 201, 108
  N Arrays:     1, actor={(0, 0): <Volume(0x55611ba1ffc0) at 0x7fda41342980>})

In [ ]:
reload(sv.plotter)
reload(sv.convenience)
reload(sv)

plotter = sv.initialize_plotter()

meshes = sv.quick_plot(
    plotter=plotter,
    simulation_ds=ds.sel({"y": slice(0, 110_000)}).where(ds["WC"] >= 1),
    # contours={"WC": {}},
    volumes={"WC": {}},
    # contours={"WC": {"isosurfaces": [2, 3, 4, 5], "cmap": "Greens", "opacity": 0.3}},
    # slices_2d={"ACCPR": {}},
)

Widget(value='<iframe src="http://localhost:44153/index.html?ui=P_0x7f1928a81e50_3&reconnect=auto" class="pyvi…

In [ ]:
reload(sv.plotter)
reload(sv.convenience)
reload(sv.core)
reload(sv)


meshes = []

# xy_stride = 25
# z_stride = 10


meshes.append(
    sv.quick_plot(
        simulation_ds=ds,
        contours={
            "WC": {"opacity": 0.3, "cmap": "Greens", "isosurfaces": [1, 3, 5, 10]}
        },
        volumes={
            "R_condensate": {
                "cmap": "Blues",
                "opacity": pv.opacity_transfer_function([20, 35, 60, 90, 250], 256),
                "clim": [1e-5, ds["R_condensate"].max().values],
            }
        },
        vectors={
            "wind": {"u": "UC", "v": "VC", "w": "WC", "factor": 500, "tolerance": 0.05}
        },
    )
)

Widget(value='<iframe src="http://localhost:42241/index.html?ui=P_0x7f76cdea4190_29&reconnect=auto" class="pyv…

In [ ]:
import pyvista as pv

pv.opacity_transfer_function([50, 100, 150, 200, 250], 5)

array([ 50, 100, 150, 200, 250])

In [ ]:
plotter.export_html("/home/cmdavis4/projects/sean_students/test_ic_export.html")

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 